# ✨FrostyFriday Week 100 - Streamlit & Apps✨

2026年5月4日―――Snowflake、超進化完了！  
非構造化データの使い方、これがニュー・スタンダードだ！  

## 準備パート

In [ ]:
%%sql -r dataframe_4
-- データベースとスキーマを選択しておく
USE DATABASE FROSTYFRIDAY_DB;
USE SCHEMA WEEK100;

In [ ]:
%%sql -r dataframe_1

-- テーブルを新規作成
CREATE OR REPLACE TABLE MEDIA_TABLE (
    id INT AUTOINCREMENT,
    description VARCHAR,
    file_col FILE
);

In [ ]:
-- サンプルデータを挿入する。TO_FILEはVALUESでは使えないので注意
-- https://docs.snowflake.com/ja/en/sql-reference/functions/to_file#known-limitations
INSERT INTO MEDIA_TABLE (description, file_col) SELECT '猫1', TO_FILE('@media_stage/cat1.png');
INSERT INTO MEDIA_TABLE (description, file_col) SELECT '猫2', TO_FILE('@media_stage/cat2.png');
INSERT INTO MEDIA_TABLE (description, file_col) SELECT '回転焼き', TO_FILE('@media_stage/ovan.png');
INSERT INTO MEDIA_TABLE (description, file_col) SELECT '子犬の動画', TO_FILE('@media_stage/puppy.mp4');

-- 今回はdescription列を個別に記載したいために上記のように書いたが、FROM DIRECTORYを使ってもいい
-- SELECT FILE_URL, TO_FILE(FILE_URL) FROM DIRECTORY(@media_stage)

## SQLによる分析

Cortex AI Multimodalを使うことで、画像や動画を分析することができる！

[Cortex AI Multimodal](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-multimodal)

In [ ]:
%%sql -r dataframe_3
-- AI_COMPLETEで画像を分析する
SELECT
    AI_COMPLETE(
        'claude-sonnet-4-5',
        '画像に何が映っているか100文字程度で説明して。',
        file_col
    ) AS analysis
FROM MEDIA_TABLE
LIMIT 1;

-- Geminiを使う場合。Geminiだと動画も扱える
-- SELECT
--     AI_COMPLETE(
--         'gemini-3.1-pro',
--         '画像に何が映っているか100文字程度で説明して。',
--         file_col
--     ) AS analysis
-- FROM MEDIA_TABLE
-- LIMIT 1;

## Pythonによる表示

ワークスペース上のNotebooksではウィジェット(ボタンやテキストボックス)の表示がまだサポートされていないので😭  
インタラクティブな操作は別途Streamlit Appを作る必要がある。  

[Notebooks in Workspaces limitations](https://docs.snowflake.com/en/user-guide/ui-snowsight/notebooks-in-workspaces/notebooks-in-workspaces-limitations)

In [ ]:
from snowflake.snowpark.context import get_active_session
from IPython.display import display, Image

session = get_active_session()

file_path_df = session.sql("""
    SELECT FL_GET_RELATIVE_PATH(file_col) AS rel_path,
           FL_GET_STAGE(file_col) AS stage_name
    FROM FROSTYFRIDAY_DB.WEEK100.MEDIA_TABLE
    WHERE id = 1
""").collect()

stage_name = file_path_df[0]["STAGE_NAME"]
rel_path = file_path_df[0]["REL_PATH"]

full_path = f"{stage_name}/{rel_path}"
file_content = session.file.get_stream(full_path, decompress=False).read()

display(Image(data=file_content))